In [ ]:
library(ggplot2)
library(dplyr)
library(tidyr)

tcga_luad_multi <- readRDS("data/TCGA/processed/TCGA_LUAD_multiomics_features.rds")
expr <- tcga_luad_multi$expr_log2
mut  <- tcga_luad_multi$mut_mat
cnv  <- tcga_luad_multi$cnv_mat
meth <- tcga_luad_multi$meth_mat
clin <- tcga_luad_multi$clinical
samples <- tcga_luad_multi$samples

expr_s <- expr[samples, , drop = FALSE]
mut_s  <- mut[samples,  , drop = FALSE]
cnv_s  <- cnv[samples,  , drop = FALSE]
meth_s <- meth[samples, , drop = FALSE]

gene <- "EGFR"

if (!gene %in% colnames(mut_s) ||
    !gene %in% colnames(expr_s) ||
    !gene %in% colnames(cnv_s)  ||
    !gene %in% colnames(meth_s)) {
  stop(paste("Gene", gene, "not found in one of the matrices; try KRAS or TP53 as an example."))
}

df_egfr <- data.frame(
  sample   = samples,
  expr     = expr_s[, gene],
  cnv      = cnv_s[, gene],
  meth     = meth_s[, gene],
  mut_stat = factor(ifelse(mut_s[, gene] == 1, "Mut", "WT"),
                    levels = c("WT","Mut")),
  stringsAsFactors = FALSE
)

## 转成长表，方便 facet_wrap
df_long <- df_egfr |>
  tidyr::pivot_longer(
    cols      = c("expr","cnv","meth"),
    names_to  = "feature",
    values_to = "value"
  )

df_long$feature <- factor(df_long$feature,
                          levels = c("expr","cnv","meth"),
                          labels = c("Expression (log2TPM)",
                                     "CNV (seg.mean)",
                                     "Methylation (β)"))

## 计算每个 feature 的 WT vs Mut 的 Wilcoxon p 值
p_df <- df_long |>
  dplyr::group_by(feature) |>
  dplyr::summarise(
    p = {
      # 确保两组都有数据
      if (length(unique(mut_stat)) < 2 ||
          any(table(mut_stat) == 0)) {
        NA_real_
      } else {
        wilcox.test(value ~ mut_stat)$p.value
      }
    },
    .groups = "drop"
  ) |>
  dplyr::mutate(
    p_label = ifelse(is.na(p), "p = NA",
                     paste0("p = ", signif(p, 2)))
  )

## 画图 + 在每个 facet 右上角加 p 值
p_egfr <- ggplot(df_long,
                 aes(x = mut_stat, y = value, fill = mut_stat)) +
  geom_boxplot(outlier.size = 0.5) +
  facet_wrap(~ feature, scales = "free_y", nrow = 1) +
  theme_bw(base_size = 11) +
  theme(
    legend.position   = "none",
    strip.background  = element_rect(fill = "grey90"),
    plot.title        = element_text(hjust = 0.5)
  ) +
  labs(
    x = "",
    y = "",
    title = paste0(gene, " multi-omic features by mutation status")
  ) +
  # 这里 y = Inf, vjust 往下挪一点，p 值贴在每个 panel 顶部
  geom_text(
    data = p_df,
    aes(x = 1.5, y = Inf, label = p_label),
    inherit.aes = FALSE,
    vjust = 1.5,
    size  = 3
  )

print(p_egfr)

dir.create("results/figures", recursive = TRUE, showWarnings = FALSE)
ggsave(paste0("results/figures/Fig1E_", gene, "_multiomic_boxplots_p.pdf"),
       plot = p_egfr, width = 5, height = 3.5)

In [ ]:
library(dplyr)

multi <- readRDS("data/TCGA/processed/TCGA_LUAD_multiomics_features.rds")

expr <- multi$expr_log2
mut  <- multi$mut_mat
cnv  <- multi$cnv_mat
meth <- multi$meth_mat
clin <- multi$clinical
samples <- multi$samples

# 统一样本顺序
expr <- expr[samples, , drop = FALSE]
mut  <- mut[samples,  , drop = FALSE]
cnv  <- cnv[samples,  , drop = FALSE]
meth <- meth[samples, , drop = FALSE]
clin <- clin[samples, , drop = FALSE]

# 生存 label
y_time  <- clin$OS.time
y_event <- clin$OS.event

# 合并特征（简单横着拼，后面在 Python 再做标准化）
X <- cbind(expr, mut, cnv, meth)

dir.create("python_data", showWarnings = FALSE)
saveRDS(list(
  X = X,
  y_time = y_time,
  y_event = y_event
), file = "python_data/tcga_luad_multi_for_python.rds")

# 也可以直接保存成 csv 和 npy，但 RDS → Python 更方便用 reticulate 或你自己再转 npy

In [ ]:
obj <- readRDS("python_data/tcga_luad_multi_for_python.rds")
X       <- obj$X
y_time  <- obj$y_time
y_event <- obj$y_event

np <- reticulate::import("numpy")
np$savez("python_data/tcga_luad_multi.npz",
         X = as.matrix(X),
         y_time  = as.numeric(y_time),
         y_event = as.numeric(y_event))

In [ ]:
tcga_luad_multi <- readRDS("data/TCGA/processed/TCGA_LUAD_multiomics_features.rds")

In [ ]:
## =========================
## Figure 1B–H master script
## =========================

library(ComplexHeatmap)
library(circlize)
library(dplyr)
library(tidyr)
library(ggplot2)
library(RColorBrewer)
library(survival)
library(survminer)
library(tibble)
library(forcats)
library(maftools)

dir.create("results/figures", showWarnings = FALSE, recursive = TRUE)

## ---- load multi-omics object ----
tcga_luad_multi <- readRDS("data/TCGA/processed/TCGA_LUAD_multiomics_features.rds")
expr <- tcga_luad_multi$expr_log2
mut  <- tcga_luad_multi$mut_mat
cnv  <- tcga_luad_multi$cnv_mat
meth <- tcga_luad_multi$meth_mat
clin <- tcga_luad_multi$clinical
samples <- tcga_luad_multi$samples   # 16 位 sample barcode

## 保证所有矩阵用同一行顺序
expr <- expr[samples, , drop = FALSE]
mut  <- mut[samples,  , drop = FALSE]
cnv  <- cnv[samples,  , drop = FALSE]
meth <- meth[samples, , drop = FALSE]
clin <- clin[samples, , drop = FALSE]

## 公用调色板
pal_set2 <- brewer.pal(8, "Set2")

## 简单 z-score helper（列）
scale_mat <- function(m) {
  m2 <- apply(m, 2, function(x) {
    z <- (x - mean(x, na.rm = TRUE)) / sd(x, na.rm = TRUE)
    z[is.na(z)] <- 0
    z
  })
  m2
}

## ════════════════════════════════
## Figure 1B  Canonical LUAD driver multi-omic heatmap
## ════════════════════════════════

# 选择一组 driver genes
driver_genes <- c("EGFR","KRAS","TP53","STK11","KEAP1",
                  "ALK","ROS1","MET")

# 只保留在各矩阵都存在的基因
driver_genes <- driver_genes[
  driver_genes %in% colnames(mut)  &
    driver_genes %in% colnames(cnv)  &
    driver_genes %in% colnames(expr) &
    driver_genes %in% colnames(meth)
]

cat("Figure 1B driver genes used:", driver_genes, "\n")

# 按 gene 子集出矩阵
mut_dr   <- mut[, driver_genes, drop = FALSE]
cnv_dr   <- cnv[, driver_genes, drop = FALSE]
expr_dr  <- expr[, driver_genes, drop = FALSE]
meth_dr  <- meth[, driver_genes, drop = FALSE]

# 表达和甲基化做 Z-score（列标准化）
expr_z <- scale_mat(expr_dr)
meth_z <- scale_mat(meth_dr)

# Mutation 用 0/1，映射颜色
mut_col_fun <- c("0" = "white", "1" = "black")
mut_mat_plot <- mut_dr
mut_mat_plot[mut_mat_plot > 1] <- 1
mut_mat_plot <- as.matrix(mut_mat_plot)

# CNV/Expr/Meth 颜色函数
cnv_col_fun <- colorRamp2(c(-1, 0, 1), c("#67a9cf", "white", "#ef8a62"))
expr_col_fun <- colorRamp2(c(-2, 0, 2), c("#2166ac", "white", "#b2182b"))
meth_col_fun <- colorRamp2(c(-2, 0, 2), c("#b2182b", "white", "#2166ac"))

ht_mut <- Heatmap(
  mut_mat_plot,
  name            = "Mutation",
  col             = mut_col_fun,
  cluster_rows    = TRUE,
  cluster_columns = FALSE,
  show_row_names  = FALSE,
  column_title    = "Canonical LUAD drivers",
  heatmap_legend_param = list(labels = c("WT","Mut"), at = c(0,1))
)

ht_cnv <- Heatmap(
  cnv_dr,
  name            = "CNV",
  col             = cnv_col_fun,
  cluster_rows    = TRUE,
  cluster_columns = FALSE,
  show_row_names  = FALSE
)

ht_expr <- Heatmap(
  expr_z,
  name            = "Expression\nz",
  col             = expr_col_fun,
  cluster_rows    = TRUE,
  cluster_columns = FALSE,
  show_row_names  = FALSE
)

ht_meth <- Heatmap(
  meth_z,
  name            = "Methylation\nz",
  col             = meth_col_fun,
  cluster_rows    = TRUE,
  cluster_columns = FALSE,
  show_row_names  = FALSE
)

ht_1B <- ht_mut + ht_cnv + ht_expr + ht_meth

# 展示
draw(ht_1B,
     heatmap_legend_side = "right",
     annotation_legend_side = "right")

# 导出
pdf("results/figures/Fig1B_driver_multiomic_heatmap.pdf", width = 6, height = 8)
draw(ht_1B,
     heatmap_legend_side = "right",
     annotation_legend_side = "right")
dev.off()

## ════════════════════════════════
## Figure 1C — patient-level expression heatmap + annotations
## ════════════════════════════════

# top 100 high-variance genes
var_all   <- apply(expr, 2, var, na.rm = TRUE)
genes_top <- names(sort(var_all, decreasing = TRUE))[1:min(100, length(var_all))]
expr_top  <- expr[, genes_top, drop = FALSE]

expr_top_z <- t(scale(t(expr_top)))
expr_top_z[is.na(expr_top_z)] <- 0
mat_heat <- t(expr_top_z)  # gene x sample

# multi-omic annotations
mut_burden <- rowSums(mut != 0, na.rm = TRUE)
cnv_thr <- 0.3
cnv_burden <- apply(cnv, 1, function(x) {
  m <- mean(abs(x) > cnv_thr, na.rm = TRUE)
  if (is.na(m)) m <- 0
  m
})

os_time <- clin$OS.time
os_group_raw <- cut(
  os_time,
  breaks = quantile(os_time, probs = c(0, 1/3, 2/3, 1), na.rm = TRUE),
  include.lowest = TRUE,
  labels = c("Low OS","Mid OS","High OS")
)
os_chr <- as.character(os_group_raw)
os_chr[is.na(os_chr)] <- "Unknown"
os_group <- factor(os_chr, levels = c("Low OS","Mid OS","High OS","Unknown"))
os_cols  <- setNames(pal_set2[1:length(levels(os_group))], levels(os_group))

ha_1C <- HeatmapAnnotation(
  `Mutation burden` = mut_burden,
  `CNV burden`      = cnv_burden,
  `OS group`        = os_group,
  col = list(
    `Mutation burden` = colorRamp2(range(mut_burden, na.rm = TRUE),
                                   c("white", pal_set2[3])),
    `CNV burden`      = colorRamp2(range(cnv_burden, na.rm = TRUE),
                                   c("white", pal_set2[4])),
    `OS group`        = os_cols
  ),
  annotation_legend_param = list(
    `Mutation burden` = list(title = "Mut burden"),
    `CNV burden`      = list(title = "CNV burden"),
    `OS group`        = list(title = "OS group")
  )
)

ht_1C <- Heatmap(
  mat_heat,
  name            = "Expression\nz",
  col             = expr_col_fun,
  top_annotation  = ha_1C,
  cluster_rows    = TRUE,
  cluster_columns = TRUE,
  show_row_names    = FALSE,
  show_column_names = FALSE,
  column_title      = "Top variable genes across TCGA-LUAD patients"
)

draw(ht_1C,
     heatmap_legend_side = "right",
     annotation_legend_side = "right")

pdf("results/figures/Fig1C_patient_expression_heatmap.pdf",
    width = 7.5, height = 6)
draw(ht_1C,
     heatmap_legend_side = "right",
     annotation_legend_side = "right")
dev.off()

## ════════════════════════════════
## Figure 1D — mutation vs CNV burden scatter
## ════════════════════════════════

df_burden <- tibble(
  sample     = samples,
  mut_burden = mut_burden,
  cnv_burden = cnv_burden,
  os_time    = os_time
)

os_group2 <- cut(
  df_burden$os_time,
  breaks = quantile(df_burden$os_time, probs = c(0, 1/3, 2/3, 1), na.rm = TRUE),
  include.lowest = TRUE,
  labels = c("Low OS","Mid OS","High OS")
)
os_group2 <- addNA(os_group2)
levels(os_group2)[is.na(levels(os_group2))] <- "NA"
df_burden$os_group <- os_group2

p1D <- ggplot(df_burden,
              aes(x = mut_burden, y = cnv_burden, colour = os_group)) +
  geom_point(alpha = 0.8, size = 1.6) +
  scale_colour_manual(values = pal_set2[1:length(levels(os_group2))]) +
  theme_bw(base_size = 12) +
  theme(
    legend.position = "right",
    plot.title      = element_text(hjust = 0.5, face = "bold")
  ) +
  labs(
    x = "Mutation burden (number of mutated genes)",
    y = "CNV burden (fraction of genes with |seg.mean| > 0.3)",
    colour = "OS group",
    title  = "Figure 1D. Mutation vs CNV burden across TCGA-LUAD"
  )

print(p1D)
ggsave("results/figures/Fig1D_mut_vs_cnv_burden.pdf",
       plot = p1D, width = 5.2, height = 4)

## ════════════════════════════════
## Figure 1E — EGFR multi-omics vs mutation status + p 值
## ════════════════════════════════

gene <- "EGFR"  # 也可改成 KRAS/TP53 等看效果

stopifnot(gene %in% colnames(mut),
          gene %in% colnames(expr),
          gene %in% colnames(cnv),
          gene %in% colnames(meth))

df_egfr <- tibble(
  sample  = samples,
  expr    = expr[, gene],
  cnv     = cnv[, gene],
  meth    = meth[, gene],
  mut_stat = factor(ifelse(mut[, gene] == 1, "Mut", "WT"),
                    levels = c("WT","Mut"))
)

df_long <- df_egfr |>
  tidyr::pivot_longer(
    cols      = c("expr","cnv","meth"),
    names_to  = "feature",
    values_to = "value"
  ) |>
  mutate(
    feature = factor(
      feature,
      levels = c("expr","cnv","meth"),
      labels = c("Expression (log2TPM)",
                 "CNV (seg.mean)",
                 "Methylation (β)")
    )
  )

p_df <- df_long |>
  group_by(feature) |>
  summarise(
    p = {
      if (length(unique(mut_stat)) < 2 || any(table(mut_stat) == 0)) NA_real_
      else wilcox.test(value ~ mut_stat)$p.value
    },
    .groups = "drop"
  ) |>
  mutate(
    p_label = ifelse(is.na(p), "p = NA",
                     paste0("p = ", signif(p, 2)))
  )

p1E <- ggplot(df_long,
              aes(x = mut_stat, y = value, fill = mut_stat)) +
  geom_boxplot(outlier.size = 0.5, width = 0.6) +
  facet_wrap(~ feature, scales = "free_y", nrow = 1) +
  scale_fill_manual(values = pal_set2[1:2]) +
  theme_bw(base_size = 11) +
  theme(
    legend.position  = "none",
    strip.background = element_rect(fill = "grey92", colour = NA),
    plot.title       = element_text(hjust = 0.5, face = "bold")
  ) +
  labs(
    x = "",
    y = "",
    title = paste0("Figure 1E. ", gene, " multi-omics by mutation status")
  ) +
  geom_text(
    data = p_df,
    aes(x = 1.5, y = Inf, label = p_label),
    inherit.aes = FALSE,
    vjust = 1.5,
    size  = 3
  )

print(p1E)
ggsave("results/figures/Fig1E_EGFR_multiomic_boxplots_p.pdf",
       plot = p1E, width = 6.5, height = 3.4)

## ════════════════════════════════
## Figure 1F — KM by mutation burden（survminer + Set2）
## ════════════════════════════════

mut_burden_all <- rowSums(mut != 0, na.rm = TRUE)

km_df <- clin |>
  mutate(mut_burden = mut_burden_all) |>
  filter(!is.na(OS.time) & !is.na(OS.event))

med_mut <- median(km_df$mut_burden, na.rm = TRUE)

km_df <- km_df |>
  mutate(
    mut_group = ifelse(mut_burden > med_mut, "High", "Low"),
    mut_group = factor(mut_group, levels = c("Low","High"))
  )

fit_mut <- survfit(Surv(OS.time, OS.event) ~ mut_group, data = km_df)

pal_km <- pal_set2[1:2]

p_km <- ggsurvplot(
  fit_mut,
  data        = km_df,
  risk.table  = TRUE,
  conf.int    = TRUE,
  conf.int.style = "ribbon",
  conf.int.alpha = 0.15,
  pval        = TRUE,
  pval.coord  = c(0, 0.08),
  pval.size   = 4,
  palette     = pal_km,
  legend.title = "Mutation burden",
  legend.labs  = c("Low", "High"),
  censor.shape = 124,
  censor.size  = 2.2,
  size         = 0.9,
  xlab        = "Time (days)",
  ylab        = "Overall survival probability",
  ggtheme     = theme_classic(base_size = 13),
  risk.table.y.text.col = TRUE,
  risk.table.y.text     = FALSE,
  risk.table.height     = 0.22,
  risk.table.fontsize   = 3
)

p_km$plot <- p_km$plot +
  theme(
    plot.title      = element_text(hjust = 0.5, face = "bold"),
    legend.position = c(0.8, 0.85),
    legend.background = element_rect(fill = NA, colour = NA),
    axis.line        = element_line(colour = "black")
  )

p_km$table <- p_km$table +
  theme_classic(base_size = 11) +
  theme(
    axis.title.x = element_blank(),
    axis.title.y = element_text(size = 10)
  )

print(p_km)

ggsave("results/figures/Fig1F_KM_mut_burden_plot_Set2.pdf",
       plot = p_km$plot, width = 5, height = 4)
ggsave("results/figures/Fig1F_KM_mut_burden_risktable_Set2.pdf",
       plot = p_km$table, width = 5, height = 2.4)

## ════════════════════════════════
## Figure 1G — Top 20 mutated genes oncoplot (maftools)
## ════════════════════════════════

maf_df <- readRDS("data/TCGA/processed/TCGA_LUAD_Masked_Somatic_Mutation.rds")

maf_df2 <- maf_df %>%
  dplyr::mutate(sample16 = substr(Tumor_Sample_Barcode, 1, 16)) %>%
  dplyr::filter(sample16 %in% samples)

maf_obj <- read.maf(maf = maf_df2, verbose = FALSE)

# 展示 oncoplot
oncoplot(maf = maf_obj, top = 20,
         sortByAnnotation = TRUE,
         draw_titv = FALSE,
         showTumorSampleBarcodes = FALSE,
         titleText = "Figure 1G. Top 20 mutated genes in TCGA-LUAD")

pdf("results/figures/Fig1G_oncoplot_top20_mutated_genes.pdf",
    width = 7, height = 6)
oncoplot(maf = maf_obj, top = 20,
         sortByAnnotation = TRUE,
         draw_titv = FALSE,
         showTumorSampleBarcodes = FALSE,
         titleText = "Top 20 mutated genes in TCGA-LUAD")
dev.off()

## ════════════════════════════════
## Figure 1H — Mutation frequency of LUAD driver genes
## ════════════════════════════════

# 多组学样本数（446 个）
n_patients <- length(samples)

gene_summ <- getGeneSummary(maf_obj)

driver_genes2 <- c("EGFR","KRAS","TP53","STK11","KEAP1",
                   "ALK","ROS1","MET")

driver_summ <- gene_summ %>%
  dplyr::filter(Hugo_Symbol %in% driver_genes2) %>%
  dplyr::select(Hugo_Symbol, MutatedSamples) %>%   # total 不用了
  dplyr::mutate(
    freq = MutatedSamples / n_patients,
    Hugo_Symbol = factor(Hugo_Symbol, levels = driver_genes2)
  )

p1H <- ggplot(driver_summ,
              aes(x = Hugo_Symbol, y = freq, fill = Hugo_Symbol)) +
  geom_col(width = 0.7) +
  scale_fill_manual(values = pal_set2[1:length(driver_genes2)]) +
  coord_cartesian(ylim = c(0, 0.6)) +   # 0–60% 更接近真实范围
  theme_bw(base_size = 12) +
  theme(
    legend.position = "none",
    axis.text.x     = element_text(angle = 30, hjust = 1),
    plot.title      = element_text(hjust = 0.5, face = "bold")
  ) +
  labs(
    x = "LUAD driver gene",
    y = "Mutation frequency\n(fraction of 446 patients)",
    title = "Figure 1H. Mutation frequency of canonical LUAD drivers"
  )

print(p1H)
ggsave("results/figures/Fig1H_driver_mutation_frequency.pdf",
       plot = p1H, width = 5.5, height = 3.5)

In [ ]:
# survival_export_perf.R

library(survival)
library(dplyr)
library(readr)

dir.create("results", showWarnings = FALSE)

tcga_luad_multi <- readRDS("data/TCGA/processed/TCGA_LUAD_multiomics_features.rds")

expr <- tcga_luad_multi$expr_log2
clin <- tcga_luad_multi$clinical
samples <- tcga_luad_multi$samples

# 对齐顺序
expr <- expr[samples, , drop = FALSE]
clin <- clin[samples, , drop = FALSE]

## 1. 构造基础 survival 数据框：OS.time / OS.event -------------------------

if (!all(c("OS.time","OS.event") %in% colnames(clin))) {
  stop("临床表中找不到 OS.time / OS.event 列，请用 colnames(clin) 检查。")
}

df_surv <- clin %>%
  transmute(
    OS.time  = OS.time,
    OS.event = OS.event
  )

# 只保留 OS 信息不缺失的样本
keep_idx <- which(!is.na(df_surv$OS.time) & !is.na(df_surv$OS.event))
df_surv   <- df_surv[keep_idx, , drop = FALSE]
expr_surv <- expr[keep_idx, , drop = FALSE]

cat("Survival cohort size after OS QC:", nrow(df_surv), "patients\n")

## 2. 动态添加 age / gender / stage，如果这些列存在且有用 -----------

predictors <- character(0)

# 年龄
if ("age_at_initial_pathologic_diagnosis" %in% colnames(clin)) {
  age_vec <- clin$age_at_initial_pathologic_diagnosis[keep_idx]
  if (sum(!is.na(age_vec)) > 0) {
    df_surv$age <- age_vec
    predictors <- c(predictors, "age")
    cat("Using age_at_initial_pathologic_diagnosis as 'age'\n")
  }
}

# 性别
if ("gender" %in% colnames(clin)) {
  gender_vec <- clin$gender[keep_idx]
  if (sum(!is.na(gender_vec)) > 0 && length(unique(na.omit(gender_vec))) > 1) {
    df_surv$gender <- factor(gender_vec)
    predictors <- c(predictors, "gender")
    cat("Using 'gender' as categorical covariate\n")
  }
}

# 分期
if ("pathologic_stage" %in% colnames(clin)) {
  stage_vec <- clin$pathologic_stage[keep_idx]
  if (sum(!is.na(stage_vec)) > 0 && length(unique(na.omit(stage_vec))) > 1) {
    df_surv$stage <- factor(stage_vec)
    predictors <- c(predictors, "stage")
    cat("Using 'pathologic_stage' as categorical covariate\n")
  }
}

cat("Clinical predictors available:", paste(predictors, collapse = ", "), "\n")

# 如果完全没有可用临床协变量，就用 null model
if (length(predictors) == 0) {
  form_clinical <- as.formula("Surv(OS.time, OS.event) ~ 1")
  cat("No usable age/gender/stage; using null CoxPH model (no covariates).\n")
} else {
  form_clinical <- as.formula(
    paste("Surv(OS.time, OS.event) ~", paste(predictors, collapse = " + "))
  )
}

## 3. CoxPH_clinical -----------------------------------------------------------

fit_clinical <- coxph(form_clinical, data = df_surv)
summary_clinical <- summary(fit_clinical)
cindex_clinical  <- as.numeric(summary_clinical$concordance[1])
cat("CoxPH_clinical C-index:", cindex_clinical, "\n")

## 4. Model 2: 临床 + 表达 PCA (PC1–PC5) --------------------------------------

var_all   <- apply(expr_surv, 2, var, na.rm = TRUE)
genes_top <- names(sort(var_all, decreasing = TRUE))[1:min(1000, length(var_all))]
expr_top  <- expr_surv[, genes_top, drop = FALSE]

expr_top_z <- scale(expr_top)
pca <- prcomp(expr_top_z, center = TRUE, scale. = FALSE)
pcs <- pca$x[, 1:5, drop = FALSE]
colnames(pcs) <- paste0("PC", 1:5)

df_surv_pcs <- cbind(df_surv, pcs)

if (length(predictors) == 0) {
  rhs <- paste(paste0("PC", 1:5), collapse = " + ")
} else {
  rhs <- paste(
    paste(predictors, collapse = " + "),
    paste(paste0("PC", 1:5), collapse = " + "),
    sep = " + "
  )
}
form_clinical_pcs <- as.formula(paste("Surv(OS.time, OS.event) ~", rhs))

fit_clinical_pcs <- coxph(form_clinical_pcs, data = df_surv_pcs)
summary_clinical_pcs <- summary(fit_clinical_pcs)
cindex_clinical_pcs  <- as.numeric(summary_clinical_pcs$concordance[1])
cat("CoxPH_clinical_exprPCs C-index:", cindex_clinical_pcs, "\n")

## 5. 生成 perf_survival_models.csv -------------------------------------------

perf_surv <- tibble::tibble(
  model  = c("CoxPH_clinical", "CoxPH_clinical_exprPCs"),
  cindex = c(cindex_clinical,  cindex_clinical_pcs),
  auroc  = c(NA_real_, NA_real_)   # 暂不算 AUROC
)

write_csv(perf_surv, "results/perf_survival_models.csv")
cat("✅ Saved results/perf_survival_models.csv\n")

In [ ]:
library(readr)
library(tibble)
dir.create("results", showWarnings = FALSE)

# perf_ablation_depmap.csv
perf_ablation <- tibble(
  feature_set = c("Expr only", "Expr+CNV", "Expr+CNV+Meth", "All+PPI"),
  metric      = "R2",
  value       = c(-0.20, -0.15, -0.13, -0.10)  # 占位数值
)
write_csv(perf_ablation, "results/perf_ablation_depmap.csv")

# perf_multitask_dep_surv.csv
multitask_gain <- tibble(
  task    = c("Dependency","Dependency","Survival","Survival"),
  setting = c("Single","Multi","Single","Multi"),
  metric  = c("R2","R2","AUROC","AUROC"),
  value   = c(-0.11, -0.08, 0.60, 0.63)  # 占位数值
)
write_csv(multitask_gain, "results/perf_multitask_dep_surv.csv")

In [ ]:
## =========================
## Figure 4 A–G: Pan-DepMap top targets in TCGA-LUAD
## =========================

library(readr)
library(dplyr)
library(tidyr)
library(ggplot2)
library(RColorBrewer)
library(survival)
library(survminer)
library(ComplexHeatmap)
library(circlize)
library(grid)
library(patchwork)

dir.create("results/figures", showWarnings = FALSE, recursive = TRUE)

## ---- 1. 读入 TCGA-LUAD 多组学 和 pan-DepMap 扫描结果 ----

tcga_luad_multi <- readRDS("data/TCGA/processed/TCGA_LUAD_multiomics_features.rds")
pan_scan        <- read_csv("results/depmap_pan_scan.csv", show_col_types = FALSE)

expr <- tcga_luad_multi$expr_log2    # 样本 x 基因，log2TPM
cnv  <- tcga_luad_multi$cnv_mat      # 样本 x 基因，seg.mean
meth <- tcga_luad_multi$meth_mat     # 样本 x 基因，promoter β，假设你存了这一项
clin <- tcga_luad_multi$clinical
samples <- tcga_luad_multi$samples

# 对齐顺序（以 expr 的样本为基准）
expr <- expr[samples, , drop = FALSE]
cnv  <- cnv [samples, , drop = FALSE]
meth <- meth[samples, , drop = FALSE]
clin <- clin[samples, , drop = FALSE]

pal_set2 <- brewer.pal(8, "Set2")

## ---- 2. 选一组 “感兴趣的基因”：pan-DepMap AUROC 排名前 8 ----

top_genes_pan <- pan_scan %>%
  filter(!is.na(AUROC)) %>%
  arrange(desc(AUROC)) %>%
  slice_head(n = 8) %>%
  pull(gene)

# 确保这些基因在 TCGA expr 里也存在
target_genes <- intersect(top_genes_pan, colnames(expr))

cat("Figure 4 使用的 target genes:\n")
print(target_genes)

if (length(target_genes) < 3) {
  warning("在 TCGA 表达矩阵里只找到少数 target genes，请考虑扩大 n 或检查基因名。")
}

## ============================================================
## 4A – Top target genes by pan-DepMap AUROC
## ============================================================

pan_top_targets <- pan_scan %>%
  group_by(gene) %>%
  summarise(
    AUROC = mean(AUROC, na.rm = TRUE),
    R2    = mean(R2, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  filter(gene %in% target_genes) %>%
  arrange(desc(AUROC))

p4A <- ggplot(pan_top_targets,
              aes(x = reorder(gene, AUROC), y = AUROC)) +
  geom_col(fill = pal_set2[1], width = 0.7) +
  coord_flip() +
  theme_bw(base_size = 12) +
  theme(
    plot.title  = element_text(hjust = 0.5, face = "bold"),
    axis.text.y = element_text(size = 10)
  ) +
  labs(
    x = "Gene",
    y = "AUROC (pan-DepMap)",
    title = "Figure 4A. Top pan-DepMap targets by AUROC"
  )

print(p4A)
ggsave("results/figures/Fig4A_pan_top_targets_AUROC.pdf",
       plot = p4A, width = 4.8, height = 4.0)

## ============================================================
## 4B – TCGA-LUAD 表达分布（这些基因的 boxplot）
## ============================================================

expr_targets <- expr[, target_genes, drop = FALSE]

expr_long <- expr_targets %>%
  as.data.frame() %>%
  tibble::rownames_to_column("sample") %>%
  pivot_longer(cols = all_of(target_genes),
               names_to = "gene", values_to = "expr")

p4B <- ggplot(expr_long,
              aes(x = gene, y = expr)) +
  geom_boxplot(fill = pal_set2[2], outlier.size = 0.4) +
  theme_bw(base_size = 12) +
  theme(
    axis.text.x  = element_text(angle = 30, hjust = 1),
    plot.title   = element_text(hjust = 0.5, face = "bold")
  ) +
  labs(
    x = "Gene",
    y = "Expression (log2 TPM)",
    title = "Figure 4B. Expression of top targets in TCGA-LUAD"
  )

print(p4B)
ggsave("results/figures/Fig4B_TCGA_expression_boxplots_targets.pdf",
       plot = p4B, width = 5.5, height = 4.0)

## ============================================================
## 4C – Expr/CNV/Meth 相关性 heatmap（per gene）
## ============================================================

# 对每个 target gene 计算：Expr vs CNV vs Meth 的相关性（Spearman）
calc_cross_omics_cor <- function(g) {
  e <- expr[, g]
  res <- tibble(
    gene = g,
    Expr_vs_CNV  = suppressWarnings(cor(e, cnv[, g],  method = "spearman", use = "pairwise.complete.obs")),
    Expr_vs_Meth = suppressWarnings(cor(e, meth[, g], method = "spearman", use = "pairwise.complete.obs")),
    CNV_vs_Meth  = suppressWarnings(cor(cnv[, g], meth[, g], method = "spearman", use = "pairwise.complete.obs"))
  )
  res
}

cors_df <- purrr::map_dfr(target_genes, calc_cross_omics_cor)

cor_mat <- cors_df %>%
  column_to_rownames("gene") %>%
  as.matrix()
cor_mat <- t(cor_mat)  # 行 = pair, 列 = gene

# 颜色函数
col_fun_cor <- colorRamp2(c(-1, 0, 1), c("#2166ac","white","#b2182b"))

ht_4C <- Heatmap(
  cor_mat,
  name = "Spearman r",
  col  = col_fun_cor,
  cluster_rows = FALSE,
  cluster_columns = FALSE,
  row_names_side = "left"
)

draw(ht_4C,
     heatmap_legend_side    = "right",
     annotation_legend_side = "right")

pdf("results/figures/Fig4C_cross_omics_correlation_targets.pdf",
    width = 6, height = 3.5)
draw(ht_4C,
     heatmap_legend_side    = "right",
     annotation_legend_side = "right")
dev.off()

## ============================================================
## 4D – 单基因 OS KM：选一个代表性基因（AUROC 高且 R² 较高）
## ============================================================

# 选一个代表性基因（score = AUROC * pmax(R2,0)，选 score 最大）
best_gene <- pan_top_targets %>%
  mutate(score = AUROC * pmax(R2, 0)) %>%
  arrange(desc(score)) %>%
  slice(1) %>%
  pull(gene)

cat("Figure 4D 使用单基因 OS 的代表性基因:", best_gene, "\n")

# 构造 survival 数据框
df_surv <- clin %>%
  transmute(
    OS.time  = OS.time,
    OS.event = OS.event,
    expr_gene = expr[, best_gene]
  )

df_surv <- df_surv %>%
  filter(!is.na(OS.time), !is.na(OS.event), !is.na(expr_gene))

# 按中位数切 high/low 组
med_expr <- median(df_surv$expr_gene, na.rm = TRUE)
df_surv <- df_surv %>%
  mutate(
    group = ifelse(expr_gene > med_expr, "High", "Low"),
    group = factor(group, levels = c("Low","High"))
  )

fit_best <- survfit(Surv(OS.time, OS.event) ~ group, data = df_surv)

p4D <- ggsurvplot(
  fit_best,
  data        = df_surv,
  risk.table  = TRUE,
  conf.int    = TRUE,
  pval        = TRUE,
  pval.coord  = c(0, 0.1),
  ggtheme     = theme_classic(base_size = 12),
  legend.title= paste0(best_gene, " expr"),
  legend.labs = c("Low","High"),
  palette     = pal_set2[3:4]
)

print(p4D)

ggsave("results/figures/Fig4D_KM_best_single_gene.pdf",
       plot = p4D$plot, width = 4.8, height = 4.0)
ggsave("results/figures/Fig4D_KM_best_single_gene_risktable.pdf",
       plot = p4D$table, width = 4.8, height = 2.5)

## ============================================================
## 4E – 多基因 signature OS KM（top target genes 的表达平均 z-score）
## ============================================================

# 计算这些基因的 z-score expression，取平均作为 signature
expr_sig <- expr[, target_genes, drop = FALSE]

expr_sig_z <- scale(expr_sig)
sig_score  <- rowMeans(expr_sig_z, na.rm = TRUE)

df_surv_sig <- clin %>%
  transmute(
    OS.time  = OS.time,
    OS.event = OS.event,
    sig      = sig_score
  ) %>%
  filter(!is.na(OS.time), !is.na(OS.event), !is.na(sig))

med_sig <- median(df_surv_sig$sig, na.rm = TRUE)

df_surv_sig <- df_surv_sig %>%
  mutate(
    group = ifelse(sig > med_sig, "High", "Low"),
    group = factor(group, levels = c("Low","High"))
  )

fit_sig <- survfit(Surv(OS.time, OS.event) ~ group, data = df_surv_sig)

p4E <- ggsurvplot(
  fit_sig,
  data        = df_surv_sig,
  risk.table  = TRUE,
  conf.int    = TRUE,
  pval        = TRUE,
  pval.coord  = c(0, 0.1),
  ggtheme     = theme_classic(base_size = 12),
  legend.title= "Multi-gene signature",
  legend.labs = c("Low","High"),
  palette     = pal_set2[5:6]
)

print(p4E)

ggsave("results/figures/Fig4E_KM_multi_gene_signature.pdf",
       plot = p4E$plot, width = 4.8, height = 4.0)
ggsave("results/figures/Fig4E_KM_multi_gene_signature_risktable.pdf",
       plot = p4E$table, width = 4.8, height = 2.5)

## ============================================================
## 4F – Scatter: pan AUROC vs TCGA OS 单基因 Cox HR
## ============================================================

# 为 target_genes 计算 TCGA 单基因 Cox HR
cox_for_gene <- function(g) {
  df <- clin %>%
    transmute(
      OS.time  = OS.time,
      OS.event = OS.event,
      expr_gene = expr[, g]
    ) %>%
    filter(!is.na(OS.time), !is.na(OS.event), !is.na(expr_gene))
  if (nrow(df) < 30) return(NA_real_)
  fit <- coxph(Surv(OS.time, OS.event) ~ expr_gene, data = df)
  hr  <- exp(coef(fit)[1])
  hr
}

hr_vec <- purrr::map_dbl(target_genes, cox_for_gene)
names(hr_vec) <- target_genes

df_hr <- tibble(
  gene = target_genes,
  HR   = hr_vec
)

df_4F <- pan_top_targets %>%
  inner_join(df_hr, by = "gene")

p4F <- ggplot(df_4F,
              aes(x = AUROC, y = HR, label = gene)) +
  geom_point(size = 2, colour = pal_set2[7]) +
  geom_text(vjust = -0.5, size = 3) +
  theme_bw(base_size = 12) +
  theme(
    plot.title  = element_text(hjust = 0.5, face = "bold")
  ) +
  labs(
    x = "AUROC (pan-DepMap)",
    y = "Hazard ratio (TCGA-LUAD, per unit expr)",
    title = "Figure 4F. Pan-DepMap dependency vs TCGA prognostic impact"
  )

print(p4F)
ggsave("results/figures/Fig4F_AUROC_vs_HR_targets.pdf",
       plot = p4F, width = 5.0, height = 4.0)

## ============================================================
## 4G – 患者 × target genes 表达 heatmap + OS group 注释
## ============================================================

# OS group 分组（按 OS.time 三分位）
os_time <- clin$OS.time
os_group_raw <- cut(
  os_time,
  breaks = quantile(os_time, probs = c(0, 1/3, 2/3, 1), na.rm = TRUE),
  include.lowest = TRUE,
  labels = c("Low OS","Mid OS","High OS")
)
os_chr <- as.character(os_group_raw)
os_chr[is.na(os_chr)] <- "Unknown"
os_group <- factor(os_chr, levels = c("Low OS","Mid OS","High OS","Unknown"))

# heatmap 表达矩阵：z-score
expr_targets_z <- t(scale(expr_targets))
expr_targets_z[is.na(expr_targets_z)] <- 0
# 行 = gene, 列 = sample
heat_mat <- expr_targets_z

os_cols <- setNames(pal_set2[1:4], levels(os_group))

ha_4G <- HeatmapAnnotation(
  `OS group` = os_group,
  col = list(`OS group` = os_cols),
  annotation_legend_param = list(`OS group` = list(title = "OS group"))
)

ht_4G <- Heatmap(
  heat_mat,
  name = "Expr z",
  col  = colorRamp2(c(-2, 0, 2), c("#2166ac","white","#b2182b")),
  top_annotation    = ha_4G,
  cluster_rows      = TRUE,
  cluster_columns   = TRUE,
  show_row_names    = TRUE,
  show_column_names = FALSE
)

draw(ht_4G,
     heatmap_legend_side    = "right",
     annotation_legend_side = "right")

pdf("results/figures/Fig4G_TCGA_heatmap_targets_OS_annotation.pdf",
    width = 7, height = 4.5)
draw(ht_4G,
     heatmap_legend_side    = "right",
     annotation_legend_side = "right")
dev.off()

## ============================================================
## 最后：用 patchwork 粗略拼一下 A–F（G 单独放）
## ============================================================

row1 <- p4A + p4B + plot_layout(ncol = 2, widths = c(1, 1.2))
row2 <- p4F + p4C$plot + plot_layout(ncol = 2, widths = c(1.2, 1))
row3 <- p4E$plot + plot_spacer() + plot_layout(ncol = 2, widths = c(1, 1))

fig4_A_F <- row1 / row2 / row3 +
  plot_layout(heights = c(1.1, 1.1, 1)) +
  plot_annotation(
    tag_levels = "A",
    tag_prefix = "",
    tag_suffix = ".",
    theme = theme(
      plot.tag = element_text(face = "bold", size = 14)
    )
  )

print(fig4_A_F)

ggsave("results/figures/Figure4_A_to_F_patchwork.pdf",
       plot = fig4_A_F, width = 11, height = 10)

In [ ]:
## =========================
## Figure 4 A–G: Pan-DepMap top targets in TCGA-LUAD (全部显示 + 保存)
## =========================

library(readr)
library(dplyr)
library(tidyr)
library(ggplot2)
library(RColorBrewer)
library(survival)
library(survminer)
library(ComplexHeatmap)
library(circlize)
library(grid)
library(tibble)
library(purrr)

dir.create("results/figures", showWarnings = FALSE, recursive = TRUE)

## ---- 1. 读入 TCGA-LUAD 多组学 + pan-DepMap 扫描 ----

tcga_luad_multi <- readRDS("data/TCGA/processed/TCGA_LUAD_multiomics_features.rds")
pan_scan        <- read_csv("results/depmap_pan_scan.csv", show_col_types = FALSE)

expr <- tcga_luad_multi$expr_log2
cnv  <- tcga_luad_multi$cnv_mat
meth <- tcga_luad_multi$meth_mat
clin <- tcga_luad_multi$clinical
samples <- tcga_luad_multi$samples

# 如果 meth_mat 名字不一致，尝试自动找一个
if (is.null(meth)) {
  if (!is.null(tcga_luad_multi$meth_genePromoter)) {
    meth <- tcga_luad_multi$meth_genePromoter
  } else if (!is.null(tcga_luad_multi$meth_gene)) {
    meth <- tcga_luad_multi$meth_gene
  } else {
    warning("未找到 meth_mat / meth_genePromoter / meth_gene，后续甲基化 panel 将全 NA。")
    meth <- matrix(NA, nrow = nrow(expr), ncol = ncol(expr),
                   dimnames = list(rownames(expr), colnames(expr)))
  }
}

# 样本顺序对齐
expr <- expr[samples, , drop = FALSE]
cnv  <- cnv [samples, , drop = FALSE]
meth <- meth[samples, , drop = FALSE]
clin <- clin[samples, , drop = FALSE]

pal_set2 <- brewer.pal(8, "Set2")

## ---- 2. 使用 TargetScore 从 pan_scan 选择 target genes ----

stopifnot(all(c("gene","AUROC","R2") %in% colnames(pan_scan)))

pan_scan <- pan_scan %>%
  mutate(
    R2_pos      = pmax(R2, 0),
    TargetScore = AUROC * R2_pos
  )

# 选 TargetScore 前 12 个基因
target_genes <- pan_scan %>%
  arrange(desc(TargetScore)) %>%
  slice_head(n = 12) %>%
  pull(gene)

# 与 TCGA 表达矩阵取交集
target_genes <- intersect(target_genes, colnames(expr))

cat("Figure 4 使用的 target genes:\n")
print(target_genes)

if (length(target_genes) < 3) {
  warning("在 TCGA expr 里匹配到的 target genes <3，某些图信息量会有限。")
}

## ============================================================
## 4A – Target genes 在 pan-DepMap 中的 AUROC 条形图
## ============================================================

target_perf <- pan_scan %>%
  filter(gene %in% target_genes) %>%
  arrange(desc(AUROC)) %>%
  select(gene, AUROC, R2)

p4A <- ggplot(target_perf,
              aes(x = reorder(gene, AUROC), y = AUROC)) +
  geom_col(fill = pal_set2[1]) +
  coord_flip() +
  theme_bw(base_size = 12) +
  theme(
    plot.title  = element_text(hjust = 0.5, face = "bold"),
    axis.text.y = element_text(size = 10)
  ) +
  labs(
    x = "Gene",
    y = "AUROC (pan-DepMap)",
    title = "Figure 4A. AUROC of selected targets in pan-DepMap"
  )

print(p4A)
ggsave("results/figures/Fig4A_pan_AUROC_selected_targets.png",
       plot = p4A, width = 5, height = 4, dpi = 300)

## ============================================================
## 4B – Target genes 在 TCGA-LUAD 的表达 boxplot
## ============================================================

expr_targets <- expr[, target_genes, drop = FALSE]

expr_long <- expr_targets %>%
  as.data.frame() %>%
  rownames_to_column("sample") %>%
  pivot_longer(cols = all_of(target_genes),
               names_to = "gene", values_to = "expr")

p4B <- ggplot(expr_long,
              aes(x = gene, y = expr)) +
  geom_boxplot(fill = pal_set2[2], outlier.size = 0.4) +
  theme_bw(base_size = 12) +
  theme(
    axis.text.x  = element_text(angle = 30, hjust = 1),
    plot.title   = element_text(hjust = 0.5, face = "bold")
  ) +
  labs(
    x = "Gene",
    y = "Expression (log2 TPM)",
    title = "Figure 4B. Expression of pan-DepMap targets in TCGA-LUAD"
  )

print(p4B)
ggsave("results/figures/Fig4B_TCGA_expression_boxplots_targets.png",
       plot = p4B, width = 6, height = 4, dpi = 300)

## ============================================================
## 4C – Expr/CNV/Meth cross-omics 相关性 heatmap
## ============================================================

calc_cross_cor <- function(g) {
  tibble(
    gene = g,
    Expr_vs_CNV  = suppressWarnings(cor(expr[, g], cnv[, g],
                                        use = "pairwise.complete.obs",
                                        method = "spearman")),
    Expr_vs_Meth = suppressWarnings(cor(expr[, g], meth[, g],
                                        use = "pairwise.complete.obs",
                                        method = "spearman")),
    CNV_vs_Meth  = suppressWarnings(cor(cnv[, g], meth[, g],
                                        use = "pairwise.complete.obs",
                                        method = "spearman"))
  )
}

cors_df <- purrr::map_dfr(target_genes, calc_cross_cor)

cor_mat <- cors_df %>%
  column_to_rownames("gene") %>%
  as.matrix()
cor_mat <- t(cor_mat)  # 行 = pair, 列 = gene

col_fun_cor <- colorRamp2(c(-1, 0, 1), c("#2166ac","white","#b2182b"))

ht_4C <- Heatmap(
  cor_mat,
  name = "Spearman r",
  col  = col_fun_cor,
  cluster_rows = FALSE,
  cluster_columns = FALSE,
  row_names_side = "left"
)

# 显示
draw(ht_4C,
     heatmap_legend_side    = "right",
     annotation_legend_side = "right")

# 保存 PNG
png("results/figures/Fig4C_cross_omics_correlation_targets.png",
    width = 1800, height = 1000, res = 200)
draw(ht_4C,
     heatmap_legend_side    = "right",
     annotation_legend_side = "right")
dev.off()

## ============================================================
## 4D – 单基因 OS KM：选 TargetScore 最高的 gene
## ============================================================

best_gene <- pan_scan %>%
  filter(gene %in% target_genes) %>%
  arrange(desc(TargetScore)) %>%
  slice(1) %>%
  pull(gene)

cat("Figure 4D 使用单基因 OS 的代表性基因:", best_gene, "\n")

df_surv_single <- clin %>%
  transmute(
    OS.time  = OS.time,
    OS.event = OS.event,
    expr_gene = expr[, best_gene]
  ) %>%
  filter(!is.na(OS.time), !is.na(OS.event), !is.na(expr_gene))

med_expr <- median(df_surv_single$expr_gene, na.rm = TRUE)

df_surv_single <- df_surv_single %>%
  mutate(
    group = ifelse(expr_gene > med_expr, "High", "Low"),
    group = factor(group, levels = c("Low","High"))
  )

fit_best <- survfit(Surv(OS.time, OS.event) ~ group, data = df_surv_single)

p4D <- ggsurvplot(
  fit_best,
  data        = df_surv_single,
  risk.table  = TRUE,
  conf.int    = TRUE,
  pval        = TRUE,
  pval.coord  = c(0, 0.1),
  ggtheme     = theme_classic(base_size = 12),
  legend.title= paste0(best_gene, " expression"),
  legend.labs = c("Low","High"),
  palette     = pal_set2[3:4]
)

print(p4D)

ggsave("results/figures/Fig4D_KM_best_single_gene.png",
       plot = p4D$plot, width = 4.8, height = 4.2, dpi = 300)
ggsave("results/figures/Fig4D_KM_best_single_gene_risktable.png",
       plot = p4D$table, width = 4.8, height = 2.5, dpi = 300)

## ============================================================
## 4E – 多基因 target signature OS KM
## ============================================================

expr_sig <- expr[, target_genes, drop = FALSE]
expr_sig_z <- scale(expr_sig)
sig_score  <- rowMeans(expr_sig_z, na.rm = TRUE)

df_surv_sig <- clin %>%
  transmute(
    OS.time  = OS.time,
    OS.event = OS.event,
    sig      = sig_score
  ) %>%
  filter(!is.na(OS.time), !is.na(OS.event), !is.na(sig))

med_sig <- median(df_surv_sig$sig, na.rm = TRUE)

df_surv_sig <- df_surv_sig %>%
  mutate(
    group = ifelse(sig > med_sig, "High", "Low"),
    group = factor(group, levels = c("Low","High"))
  )

fit_sig <- survfit(Surv(OS.time, OS.event) ~ group, data = df_surv_sig)

p4E <- ggsurvplot(
  fit_sig,
  data        = df_surv_sig,
  risk.table  = TRUE,
  conf.int    = TRUE,
  pval        = TRUE,
  pval.coord  = c(0, 0.1),
  ggtheme     = theme_classic(base_size = 12),
  legend.title= "Target signature",
  legend.labs = c("Low","High"),
  palette     = pal_set2[5:6]
)

print(p4E)

ggsave("results/figures/Fig4E_KM_multi_gene_signature.png",
       plot = p4E$plot, width = 4.8, height = 4.2, dpi = 300)
ggsave("results/figures/Fig4E_KM_multi_gene_signature_risktable.png",
       plot = p4E$table, width = 4.8, height = 2.5, dpi = 300)

## ============================================================
## 4F – AUROC (pan-DepMap) vs hazard ratio (TCGA-LUAD)
## ============================================================

cox_for_gene <- function(g) {
  df <- clin %>%
    transmute(
      OS.time  = OS.time,
      OS.event = OS.event,
      expr_gene = expr[, g]
    ) %>%
    filter(!is.na(OS.time), !is.na(OS.event), !is.na(expr_gene))
  if (nrow(df) < 30) return(NA_real_)
  fit <- coxph(Surv(OS.time, OS.event) ~ expr_gene, data = df)
  hr  <- exp(coef(fit)[1])
  hr
}

hr_vec <- purrr::map_dbl(target_genes, cox_for_gene)
names(hr_vec) <- target_genes

df_hr <- tibble(
  gene = target_genes,
  HR   = hr_vec
)

df_4F <- target_perf %>% inner_join(df_hr, by = "gene")

p4F <- ggplot(df_4F,
              aes(x = AUROC, y = HR, label = gene)) +
  geom_point(size = 2, colour = pal_set2[7]) +
  geom_text(vjust = -0.6, size = 3) +
  theme_bw(base_size = 12) +
  theme(
    plot.title  = element_text(hjust = 0.5, face = "bold")
  ) +
  labs(
    x = "AUROC (pan-DepMap)",
    y = "Hazard ratio (per unit expression)",
    title = "Figure 4F. Dependency AUROC vs prognostic impact"
  )

print(p4F)
ggsave("results/figures/Fig4F_AUROC_vs_HR_targets.png",
       plot = p4F, width = 5.5, height = 4, dpi = 300)

## ============================================================
## 4G – 患者 × target genes 表达 heatmap + OS group 注释
## ============================================================

os_time <- clin$OS.time
os_group_raw <- cut(
  os_time,
  breaks = quantile(os_time, probs = c(0, 1/3, 2/3, 1), na.rm = TRUE),
  include.lowest = TRUE,
  labels = c("Low OS","Mid OS","High OS")
)
os_chr <- as.character(os_group_raw)
os_chr[is.na(os_chr)] <- "Unknown"
os_group <- factor(os_chr, levels = c("Low OS","Mid OS","High OS","Unknown"))

expr_targets_z <- t(scale(expr_targets))
expr_targets_z[is.na(expr_targets_z)] <- 0
heat_mat <- expr_targets_z

os_cols <- setNames(pal_set2[1:4], levels(os_group))

ha_4G <- HeatmapAnnotation(
  `OS group` = os_group,
  col = list(`OS group` = os_cols),
  annotation_legend_param = list(`OS group` = list(title = "OS group"))
)

ht_4G <- Heatmap(
  heat_mat,
  name = "Expr z",
  col  = colorRamp2(c(-2, 0, 2), c("#2166ac","white","#b2182b")),
  top_annotation    = ha_4G,
  cluster_rows      = TRUE,
  cluster_columns   = TRUE,
  show_row_names    = TRUE,
  show_column_names = FALSE
)

draw(ht_4G,
     heatmap_legend_side    = "right",
     annotation_legend_side = "right")

png("results/figures/Fig4G_TCGA_heatmap_targets_OS_annotation.png",
    width = 1800, height = 1000, res = 200)
draw(ht_4G,
     heatmap_legend_side    = "right",
     annotation_legend_side = "right")
dev.off()

In [ ]:
# 1. 读 RDS
tcga_luad_multi <- readRDS("data/TCGA/processed/TCGA_LUAD_multiomics_features.rds")

# 2. 拿出几个关键矩阵/表
expr_log2  <- tcga_luad_multi$expr_log2              # 样本 × 基因
cnv_mat    <- tcga_luad_multi$cnv_mat                # 样本 × 基因
meth_mat   <- tcga_luad_multi$meth_mat               # 样本 × 基因 promoter β
clin_level <- tcga_luad_multi$clinical               # 样本 × 临床信息（包含 OS.time, OS.event）

# 3. 确保目录存在
dir.create("data/TCGA/processed", showWarnings = FALSE, recursive = TRUE)

# 4. 导出为 CSV（行=样本，列=基因/临床变量）
write.csv(expr_log2,
          file = "data/TCGA/processed/TCGA_LUAD_expr_log2.csv",
          quote = TRUE)
write.csv(cnv_mat,
          file = "data/TCGA/processed/TCGA_LUAD_CNV_geneTSS_matrix.csv",
          quote = TRUE)
write.csv(meth_mat,
          file = "data/TCGA/processed/TCGA_LUAD_Methylation_genePromoter_matrix.csv",
          quote = TRUE)
write.csv(clin_level,
          file = "data/TCGA/processed/TCGA_LUAD_Clinical_sample_level.csv",
          quote = TRUE)



In [ ]:
project_root <- normalizePath(getwd())
if (!file.exists(file.path(project_root, "DATA_DICTIONARY.md"))) {
  stop("Please start R/Jupyter from the repository root before running this notebook.")
}
setwd(project_root)
cat("Using project root:", project_root, "
")

In [ ]:
library(maftools)

# 读入你现有的 MAF rds
maf_rds_path <- "data/TCGA/processed/TCGA_LUAD_Masked_Somatic_Mutation.rds"
luad_maf <- readRDS(maf_rds_path)

luad_maf

In [ ]:
# 读入这个 rds
maf_rds_path <- "data/TCGA/processed/TCGA_LUAD_Masked_Somatic_Mutation.rds"
maf_df <- readRDS(maf_rds_path)   # 这是一个 spec_tbl_df / tibble

class(maf_df)
colnames(maf_df)[1:20]  # 看一下前面几列的名字

In [ ]:
# 确保列名存在
colnames(maf_df)

# 如果有 "Tumor_Sample_Barcode" 和 "Hugo_Symbol"：
m <- maf_df[, c("Tumor_Sample_Barcode", "Hugo_Symbol")]

# 如果列名略有不同，比如 "Tumor_Sample_Barcode" 是 "Tumor_Sample_Barcode" 而 gene 是 "Hugo_Symbol"
# 你只要把上面的名字改成实际的列名即可。

In [ ]:
m <- subset(m, Hugo_Symbol %in% c("EGFR", "KRAS"))

In [ ]:
out_csv <- "data/TCGA/processed/TCGA_LUAD_Masked_Somatic_Mutation.csv"
dir.create("data/TCGA/processed", showWarnings = FALSE, recursive = TRUE)

# 直接把 maf_df 整个写出去（所有列、所有基因）
write.csv(maf_df, file = out_csv, row.names = FALSE)

out_csv

In [ ]:
eset <- gset[[1]]  # ExpressionSet
expr_mat <- exprs(eset)  # matrix: probes × samples
dim(expr_mat)
head(rownames(expr_mat))

In [ ]:
project_root <- normalizePath(getwd())
if (!file.exists(file.path(project_root, "DATA_DICTIONARY.md"))) {
  stop("Please start R/Jupyter from the repository root before running this notebook.")
}
setwd(project_root)
cat("Using project root:", project_root, "
")